**Verifikasi GPU**

In [ ]:
# Memeriksa ketersediaan GPU NVIDIA di Colab
import torch
print("CUDA tersedia:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU belum aktif! Runtime > Change runtime type > A100 / T4 GPU, lalu jalankan ulang dari sel 1."
print(torch.cuda.get_device_name(0))


Menghubungkan ke Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DIVINA_ORAGEN'
print("Folder Drive ditemukan:", os.path.exists(DRIVE_ROOT))
print("ORAGEN.zip ada:", os.path.exists(f'{DRIVE_ROOT}/ORAGEN.zip'))


Ambil kode ORAGEN (GitHub) & ekstrak dataset

In [ ]:
import os, glob, subprocess

# --- 1. Ambil / perbarui kode ORAGEN dari GitHub ---
!rm -rf /content/ORAGEN
!git clone https://github.com/Brata24/oragen-divina.git /content/ORAGEN

# --- 2. Ekstrak dataset BRAVE-MASKS (ORAGEN.zip) ---
!rm -rf /content/dataset && mkdir -p /content/dataset
assert os.path.exists(f'{DRIVE_ROOT}/ORAGEN.zip'), "ORAGEN.zip tidak ada di Drive/DIVINA_ORAGEN"
!unzip -q {DRIVE_ROOT}/ORAGEN.zip -d /content/dataset/

data_root = None
for root, dirs, files in os.walk('/content/dataset'):
    if 'audio' in dirs and 'video' in dirs and 'divina' not in root:
        data_root = root
        break
assert data_root, "Folder audio/video BRAVE-MASKS tidak ditemukan di dalam ORAGEN.zip"
if data_root != '/content/dataset/shared':
    if os.path.exists('/content/dataset/shared'):
        !rm -rf /content/dataset/shared
    os.symlink(data_root, '/content/dataset/shared')
print("Dataset BRAVE-MASKS root:", data_root, "-> /content/dataset/shared")

# --- 3. Ekstrak dataset DIVINA (DIVINA.zip) ---
divina_zip = f'{DRIVE_ROOT}/DIVINA.zip'
if os.path.exists(divina_zip):
    !rm -rf /content/dataset/divina && mkdir -p /content/dataset/divina
    !unzip -q -o {divina_zip} -d /content/dataset/divina/
    print("Dataset DIVINA berhasil diekstrak ke /content/dataset/divina")
else:
    print("Catatan: DIVINA.zip belum ada di Drive/DIVINA_ORAGEN (hanya menjalankan Skenario A)")

# --- 4. Label & VAD metadata ---
os.makedirs('/content/ORAGEN/data', exist_ok=True)
for fn in ['bravemasks_labels.csv', 'bravemasks_vad.pickle', 'divina_labels.csv', 'divina_vad.pickle']:
    dst = f'/content/ORAGEN/data/{fn}'
    if not os.path.exists(dst) and os.path.exists(f'{DRIVE_ROOT}/{fn}'):
        !cp {DRIVE_ROOT}/{fn} {dst}
    print(fn, "ada:", os.path.exists(dst))


In [ ]:
# Memeriksa isi folder DIVINA_ORAGEN di Google Drive untuk melihat daftar file yang ada
import os

drive_path = '/content/drive/MyDrive/DIVINA_ORAGEN'
if os.path.exists(drive_path):
    print(f"Isi folder {drive_path}:")
    files = os.listdir(drive_path)
    if files:
        for f in files:
            print(f" - {f}")
    else:
        print("Folder kosong!")
else:
    print(f"Folder {drive_path} tidak ditemukan di Drive Anda.")

Install library

In [ ]:
!pip install -q librosa soundfile soxr tensorboard

Patch kompatibilitas `transformers` versi baru


In [ ]:
import re, os

def patch_file(path, class_names):
    with open(path, 'r') as f:
        src = f.read()
    # buang atribut lama (versi list) bila ada
    src = re.sub(r"\n[ ]+_tied_weights_keys = \[\]\n[ ]+all_tied_weights_keys = (\[\]|\{\})\n", "\n", src)
    for cls in class_names:
        header = re.search(rf"class {cls}\(.*?\):\n", src)
        if header:
            src = src[:header.end()] + "    _tied_weights_keys = []\n    all_tied_weights_keys = {}\n\n" + src[header.end():]
    with open(path, 'w') as f:
        f.write(src)
    print("Patched:", path, class_names)

base = '/content/ORAGEN/src'
patch_file(f'{base}/audio/models/audio_models.py', ['AGenderAudioW2V2Model', 'AGenderAudioHuBERTModel'])
patch_file(f'{base}/fusion/models/models_wrappers.py', ['AudioWrapperModel'])

Tulis konfigurasi path Colab

In [ ]:
import os

config_dir = '/content/ORAGEN/src/configs'
os.makedirs(config_dir, exist_ok=True)

config_code = """import sys
sys.path.append('/content/ORAGEN/src')

BRAVEMASKS_RAW = '/content/dataset/shared'
DIVINA_RAW     = '/content/dataset/divina'

data_config = {
    'BRAVEMASKS': {
        'RAW_DATA_ROOT': BRAVEMASKS_RAW,
        'PROCESSED_DATA_ROOT': BRAVEMASKS_RAW,
        'VAD_FILE_PATH': '/content/ORAGEN/data/bravemasks_vad.pickle',
        'LABELS_FILE_PATH': '/content/ORAGEN/data/bravemasks_labels.csv',
        'C_NAMES': {
            'gen': ['female', 'male'],
            'mask': ['No mask', 'Tissue mask', 'Medical mask', 'Protective mask (ffp2/ffp3)',
                     'Respirator', 'Protective face shield', 'Hijab', 'Helmet', 'Glasses', 'Hat'],
        },
    },
    'DIVINA': {
        'RAW_DATA_ROOT': DIVINA_RAW,
        'PROCESSED_DATA_ROOT': DIVINA_RAW,
        'VAD_FILE_PATH': '/content/ORAGEN/data/divina_vad.pickle',
        'LABELS_FILE_PATH': '/content/ORAGEN/data/divina_labels.csv',
        'C_NAMES': {
            'gen': ['female', 'male'],
            'mask': ['No mask', 'Tissue mask', 'Medical mask', 'Protective mask (ffp2/ffp3)',
                     'Respirator', 'Protective face shield', 'Hijab', 'Helmet', 'Glasses', 'Hat'],
        },
    },
}

training_config = {
    'LOGS_ROOT': '/content/drive/MyDrive/DIVINA_ORAGEN/logs',
    'MODEL': {'cls': None, 'args': {}},
    'FEATURE_EXTRACTOR': {
        'FEATURES_ROOT': '/content/drive/MyDrive/DIVINA_ORAGEN/features',
        'FEATURES_FILE_NAME': 'SAMPLES',
        'WIN_MAX_LENGTH': 4,
        'WIN_SHIFT': 2,
        'WIN_MIN_LENGTH': 1,
        'SR': 16000,
    },
    'NUM_EPOCHS': 500,
    'BATCH_SIZE': 16,
    # Early stopping memantau measure utama (age_MAE, satuan tahun) pada subset evaluasi
    'EARLY_STOPPING': {'patience': 20, 'min_delta': 0.05},
    'OPTIMIZER': {'cls': 'AdamW', 'lr': 1e-4},
    'SCHEDULER': {'cls': 'CosineAnnealingWarmRestarts', 'T_0': 10, 'T_mult': 2},
    'AUGMENTATION': True,
    'USE_AMP': True,
}
"""

with open(f'{config_dir}/default_config.py', 'w') as f:
    f.write(config_code)
for alias in ['multimodal_bm_config.py', 'unimodal_bm_config.py', 'unimodal_config.py', 'multimodal_config.py']:
    with open(f'{config_dir}/{alias}', 'w') as f:
        f.write("from configs.default_config import data_config, training_config\n")
print("Konfigurasi Skenario A & B ditulis.")


Verifikasi sebelum training

In [ ]:
import os, pandas as pd

checks = {
    'run_train': '/content/ORAGEN/src/fusion/run_train_multimodal_maskagender.py',
    'labels': '/content/ORAGEN/data/bravemasks_labels.csv',
    'vad': '/content/ORAGEN/data/bravemasks_vad.pickle',
    'audio dir': '/content/dataset/shared/audio',
    'video dir': '/content/dataset/shared/video',
}
for k, v in checks.items():
    print(f"{k:10s}", "OK " if os.path.exists(v) else "HILANG", v)

df = pd.read_csv('/content/ORAGEN/data/bravemasks_labels.csv', low_memory=False)
row = df.iloc[0]
print("Contoh audio ada:", os.path.exists('/content/dataset/shared/' + row['audio_file_path']))
print("Contoh gambar ada:", os.path.exists('/content/dataset/shared/' + row['image_file_path']))
print(df['subset'].value_counts())

In [ ]:
import os

dataset_file = '/content/ORAGEN/src/fusion/data/agender_multimodal_features_dataset.py'
# Jika runtime ter-reset, folder ORAGEN hilang -> jalankan ulang sel clone (sel 5) s/d sel config dulu
assert os.path.exists(dataset_file), 'ORAGEN belum ada. Jalankan ulang dari sel clone/ekstrak dataset (sel 5) sampai sel config, baru sel ini.'

code = open(dataset_file, 'r', encoding='utf-8').read().replace('\r\n', '\n').replace('self.full_features_path, \n', 'self.full_features_path,\n')

# Lewati file audio yang tidak ada di disk
old1 = """            sample_fp = os.path.join(self.data_root, sample_filename)
"""
new1 = old1 + """            if not os.path.exists(sample_fp):
                continue
"""
if 'if not os.path.exists(sample_fp):' not in code:
    assert old1 in code, 'Target patch 1 tidak ditemukan'
    code = code.replace(old1, new1, 1)

# Resume cache: jangan ekstrak ulang fitur yang sudah ada
old2 = """                features = {'acoustic_features': audio_feature_extractor(wave), 'visual_features': video_feature_extractor(images_fp)}
"""
new2 = """                feature_target_path = os.path.join(self.full_features_path, sample_filename.replace('.wav', '_{0}.dat'.format(w_idx)))
                features = None
                if not os.path.exists(feature_target_path):
                    features = {'acoustic_features': audio_feature_extractor(wave), 'visual_features': video_feature_extractor(images_fp)}
"""
old3 = """                save_data(features, os.path.join(self.full_features_path,
                                                 sample_filename.replace('.wav', '_{0}.dat'.format(w_idx))))"""
new3 = """                if features is not None:
                    save_data(features, feature_target_path)"""
if 'feature_target_path' not in code:
    assert old2 in code and old3 in code, 'Target patch 2 tidak ditemukan'
    code = code.replace(old2, new2, 1).replace(old3, new3, 1)

open(dataset_file, 'w', encoding='utf-8').write(code)
import py_compile; py_compile.compile(dataset_file, doraise=True)
print('Patch OK (skip file hilang + resume cache)')


Patch: 500 epoch + Early Stopping

In [ ]:
# ==== PATCH: 500 epoch + Early Stopping (patience=10, min_delta dari config) ====
import os, py_compile

BASE = '/content/ORAGEN/src'
trainer_fp = f'{BASE}/common/net_trainer/multitask_net_trainer.py'
runner_fp  = f'{BASE}/fusion/run_train_multimodal_maskagender.py'
assert os.path.exists(trainer_fp), 'ORAGEN belum ada. Jalankan ulang dari sel clone (sel 5).'

def rd(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')
def wr(p, s):
    open(p, 'w', encoding='utf-8').write(s)
    py_compile.compile(p, doraise=True)

# ---------- 1) Trainer: tambah logika early stopping ----------
t = rd(trainer_fp)
if 'EarlyStopping' not in t:
    a1 = "            verbose: bool = True) -> None:\n        \"\"\"Iterates over epochs"
    assert a1 in t, 'anchor 1 tidak ditemukan'
    t = t.replace(a1, "            verbose: bool = True,\n            early_stopping: dict = None) -> None:\n        \"\"\"Iterates over epochs", 1)

    a2 = "        for epoch in range(1, num_epochs + 1):\n"
    assert t.count(a2) == 1, 'anchor 2 tidak unik'
    t = t.replace(a2, "        es_best, es_best_epoch, es_counter = None, 0, 0\n" + a2, 1)

    a3 = "            d_epoch_stats = {'epoch': epoch}\n"
    assert t.count(a3) == 1, 'anchor 3 tidak unik'
    t = t.replace(a3, a3 + "            es_values = []\n", 1)

    a4 = "                    performance = self.calc_metrics(targets, predicts, sample_info, db)\n"
    assert t.count(a4) == 1, 'anchor 4 tidak unik'
    t = t.replace(a4, a4 + "                    if ('train' not in phase) and (main_measure_name in performance):\n"
                           "                        es_values.append(performance[main_measure_name])\n", 1)

    a5 = "                                   sep=';', index=False)\n"
    assert t.count(a5) == 1, 'anchor 5 tidak unik'
    es_block = (
        "\n"
        "            # EarlyStopping: pantau measure utama (measures[0]) pada fase evaluasi\n"
        "            if early_stopping and es_values:\n"
        "                es_mode = self.measures[0].mode\n"
        "                es_cur = float(np.mean(es_values))\n"
        "                es_delta = float(early_stopping.get('min_delta', 0.0))\n"
        "                es_patience = int(early_stopping.get('patience', 10))\n"
        "                if es_best is None:\n"
        "                    es_improved = True\n"
        "                elif es_mode == MeasureMode.LESS:\n"
        "                    es_improved = es_cur < es_best - es_delta\n"
        "                else:\n"
        "                    es_improved = es_cur > es_best + es_delta\n"
        "                if es_improved:\n"
        "                    es_best, es_best_epoch, es_counter = es_cur, epoch, 0\n"
        "                else:\n"
        "                    es_counter += 1\n"
        "                es_msg = 'EarlyStopping: {0}={1:.4f} | best={2:.4f} (epoch {3}) | counter {4}/{5}'.format(\n"
        "                    main_measure_name, es_cur, es_best, es_best_epoch, es_counter, es_patience)\n"
        "                self.logger.info(es_msg)\n"
        "                if es_counter >= es_patience:\n"
        "                    self.logger.info('Early stopping di epoch {0}. Epoch terbaik: {1}'.format(epoch, es_best_epoch))\n"
        "                    break\n"
    )
    t = t.replace(a5, a5 + es_block, 1)
    wr(trainer_fp, t)

# ---------- 2) Runner: baca NUM_EPOCHS & EARLY_STOPPING dari config ----------
r = rd(runner_fp)
if "early_stopping=t_config.get('EARLY_STOPPING')" not in r:
    b1 = "log_epochs=[i for i in range(0, 10)])"
    assert r.count(b1) == 1, 'anchor runner 1 tidak unik'
    r = r.replace(b1, "log_epochs=[i for i in range(0, 10)],\n                                      early_stopping=t_config.get('EARLY_STOPPING'))", 1)
r = r.replace("t_config['NUM_EPOCHS'] = 100", "t_config['NUM_EPOCHS'] = t_config.get('NUM_EPOCHS', 500)")
wr(runner_fp, r)

print('Patch OK -> NUM_EPOCHS & EARLY_STOPPING diambil dari config (sel konfigurasi)')


Jalankan Training ORAGEN

In [ ]:
%cd /content/ORAGEN
!python src/fusion/run_train_multimodal_maskagender.py

Visualisasi hasil training (kurva loss/metrik, confusion matrix, distribusi data)

In [ ]:
# ==== VISUALISASI HASIL TRAINING (untuk Bab 4 TA) ====
import os, glob, shutil
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, display

LOGS = '/content/drive/MyDrive/DIVINA_ORAGEN/logs'
FIG  = '/content/drive/MyDrive/DIVINA_ORAGEN/figures'
os.makedirs(FIG, exist_ok=True)

# Ambil eksperimen terbaru (atau isi manual: EXP = f'{LOGS}/<nama_folder>')
exps = sorted([d for d in glob.glob(f'{LOGS}/*') if os.path.exists(f'{d}/logs/stats.csv')], key=os.path.getmtime)
assert exps, 'Belum ada hasil training di folder logs'
EXP = exps[-1]
NAME = os.path.basename(EXP)
print('Eksperimen:', NAME)

df = pd.read_csv(f'{EXP}/logs/stats.csv', sep=';')

# ---------- 1) Learning curve: loss train vs evaluasi ----------
loss_cols = [c for c in df.columns if c.endswith('_loss')]
plt.figure(figsize=(7, 4))
for c in loss_cols:
    plt.plot(df['epoch'], df[c], label=c.replace('_loss', ''))
plt.xlabel('Epoch'); plt.ylabel('Loss total'); plt.title('Kurva Loss'); plt.grid(alpha=.3); plt.legend()
plt.tight_layout(); plt.savefig(f'{FIG}/{NAME}_loss.png', dpi=300); plt.show()

# ---------- 2) Kurva metrik per epoch + epoch terbaik ----------
eval_mae = [c for c in df.columns if c.endswith('age_MAE') and '_train_' not in c]
best_epoch = int(df.loc[df[eval_mae].mean(axis=1).idxmin(), 'epoch']) if eval_mae else None
print('Epoch terbaik (MAE evaluasi minimum):', best_epoch)

metrics = [('age_MAE', 'MAE usia (tahun)'), ('gen_UAR', 'UAR gender (%)'), ('mask_UAR', 'UAR oklusi (%)')]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (m, lbl) in zip(axes, metrics):
    for c in [c for c in df.columns if c.endswith(m)]:
        ax.plot(df['epoch'], df[c], label=c.replace('_' + m, ''))
    if best_epoch:
        ax.axvline(best_epoch, color='red', ls='--', lw=1, label=f'epoch terbaik ({best_epoch})')
    ax.set_xlabel('Epoch'); ax.set_ylabel(lbl); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/{NAME}_metrics.png', dpi=300); plt.show()

# ---------- 3) Ringkasan metrik pada epoch terbaik ----------
if best_epoch:
    row = df[df['epoch'] == best_epoch].T
    row.columns = [f'epoch_{best_epoch}']
    display(row)
    row.to_csv(f'{FIG}/{NAME}_best_epoch_metrics.csv')

# ---------- 4) Confusion matrix (gender & oklusi) pada epoch terbaik ----------
cms = sorted(glob.glob(f'{EXP}/models/**/epoch_{best_epoch}_*.svg', recursive=True)) if best_epoch else []
seen = set()
for fp in cms:
    key = '_'.join(os.path.basename(fp).split('_')[:-1])  # 1 file per task/fase
    if key in seen:
        continue
    seen.add(key)
    shutil.copy(fp, FIG)
    print(os.path.basename(fp)); display(SVG(fp))

# ---------- 5) Distribusi usia & gender dataset (seperti Fig. 6 artikel ORAGEN) ----------
lab = pd.read_csv('/content/ORAGEN/data/bravemasks_labels.csv', low_memory=False).drop_duplicates('audio_file_path')
subsets = [s for s in ['train', 'test'] if s in lab['subset'].unique()]
fig, axes = plt.subplots(1, len(subsets), figsize=(6 * len(subsets), 4), squeeze=False)
for ax, sub in zip(axes[0], subsets):
    d = lab[lab['subset'] == sub]
    for g in sorted(d['gender'].dropna().astype(str).unique()):
        ages = pd.to_numeric(d[d['gender'].astype(str) == g]['age'], errors='coerce').dropna()
        ax.hist(ages, bins=range(0, 101, 5), alpha=.6, label=g)
    ax.set_title(f'BRAVE-MASKS - {sub}'); ax.set_xlabel('Usia (tahun)'); ax.set_ylabel('Jumlah file'); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIG}/bravemasks_age_gender_distribution.png', dpi=300); plt.show()

print('Semua gambar tersimpan di:', FIG)


### Evaluasi Zero-Shot Model Baseline (Skenario A) pada Data DIVINA (Indonesia)
Sesuai Proposal Bab 3.8.1: Menguji model baseline yang hanya dilatih data Rusia pada data uji orang Indonesia (`test_divina`) untuk mengukur besarnya dampak **Domain Shift** sebelum model diperluas.

In [ ]:
# ==== EVALUASI ZERO-SHOT MODEL BASELINE PADA TEST DIVINA ====
%cd /content/ORAGEN
!python src/fusion/run_eval_zeroshot_divina.py --logs_root /content/drive/MyDrive/DIVINA_ORAGEN/logs --subset test


### Pelatihan Model Diperluas (Skenario B: BRAVE-MASKS + DIVINA Train)
Melatih ulang blok MTCMA ORAGEN dengan gabungan data latihan internasional + data latihan Indonesia (10 kelas oklusi, 500 epoch dengan Early Stopping).

In [ ]:
# ==== JALANKAN TRAINING SKENARIO B ====
%cd /content/ORAGEN
!python src/fusion/run_train_multimodal_maskagender.py --scenario B
